# SDT full-parameter DPO: scalable training, validation tuning, and locked testing

This notebook supports both the supplied structural sample and a larger SDT dataset.
It prepares auditable preference pairs, optionally compares hyperparameter configurations
using **validation only**, freezes one selected checkpoint, and opens the test split only
when explicitly enabled.

For a full production run, update the data path and preparation settings below. Run the
cells in order in a fresh GPU runtime. Do not use test results to choose hyperparameters.


## 1. Confirm that Colab assigned a GPU
Select **Runtime → Change runtime type → GPU** before running this cell.

In [ ]:
import subprocess
import torch

subprocess.run(["nvidia-smi"], check=True)
assert torch.cuda.is_available(), "No CUDA GPU is available. Change the Colab runtime to GPU."
print("GPU:", torch.cuda.get_device_name(0))

## 2. Clone or update the repository
For a private repository, create a Colab secret named `GITHUB_TOKEN`, paste a GitHub token with read access, and enable notebook access to that secret. A public repository needs no token.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Rana-Ezzeddine/SDT.git"
REPO_DIR = Path("/content/SDT")

github_token = None
try:
    from google.colab import userdata
    github_token = userdata.get("GITHUB_TOKEN")
except Exception:
    pass

git = ["git"]
if github_token:
    git += ["-c", f"http.extraHeader=AUTHORIZATION: bearer {github_token}"]

if not REPO_DIR.exists():
    try:
        subprocess.run(git + ["clone", REPO_URL, str(REPO_DIR)], check=True)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            "Clone failed. If the repository is private, add a GITHUB_TOKEN Colab secret."
        ) from exc
elif (REPO_DIR / ".git").exists():
    subprocess.run(git + ["-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository. Use a fresh runtime.")

os.chdir(REPO_DIR)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("Working directory:", Path.cwd())
subprocess.run(["git", "status", "--short", "--branch"], check=True)

## 2b. Verify that the checked-in trainer and evaluator fixes are present

The notebook now fails fast when the repository is stale instead of rewriting source files
inside Colab. Pull the latest commit if an assertion fails.


In [ ]:
from pathlib import Path

train_code = Path("src/sdt_dpo/train.py").read_text()
evaluate_code = Path("src/sdt_dpo/evaluate.py").read_text()

assert 'trainer.evaluate(metric_key_prefix="validation")' in train_code, (
    "The trainer compatibility fix is missing. Pull the latest repository commit."
)
assert "warmup_steps" in train_code and "warmup_ratio" not in train_code, (
    "The checked-in warm-up configuration is stale. Pull the latest commit."
)
assert "return_offsets_mapping=True" in evaluate_code, (
    "The Qwen response-boundary fix is missing. Pull the latest commit."
)
assert "chat_template_prefix_mismatch" not in evaluate_code, (
    "The old evaluator is still present. Pull the latest commit."
)
assert "chosen_logprob_sum" in evaluate_code, (
    "The evaluator does not provide DPO-relative comparison fields."
)
print("Checked-in trainer and evaluator fixes confirmed.")


## 3. Install the project
The public Qwen checkpoint will be downloaded automatically later; no Hugging Face token is required.

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True)

## 4. Configure the dataset and pair-preparation policy

Set `RAW_DATA_PATH` to the complete JSON file for a full run. The default 80/10/10
prompt split is more suitable for a large dataset; the earlier 70/15/15 sample split was
chosen only to leave more sample prompts for validation and testing.

The current pair builder groups normalized exact prompts. For production, semantic
near-duplicate clustering and metadata stratification should be completed upstream before
this final split is treated as leakage-safe.


In [ ]:
import hashlib
import json
from pathlib import Path

# Change this path when the complete dataset is available.
RAW_DATA_PATH = Path("data/raw/sdt_100_llama.json")
PAIRS_PATH = Path("data/processed/dpo_pairs.jsonl")
PAIR_REPORT_PATH = Path("data/processed/pair_report.json")

# Large-dataset defaults. For exact reproduction of the old 100-record sample,
# use TRAIN_SHARE = 0.70 and VALIDATION_SHARE = 0.15.
TRAIN_SHARE = 0.80
VALIDATION_SHARE = 0.10
TEST_SHARE = 1.0 - TRAIN_SHARE - VALIDATION_SHARE

MIN_COMMON_JUDGES = 2
MIN_MARGIN = 0.10
MIN_CONFIDENCE = 0.60
SCORE_SPAN = 4.0       # 1-to-5 scoring scale: 5 - 1 = 4
SEED = 42

BASELINE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
BASE_TRAIN_CONFIG = Path("configs/full.yaml")

assert RAW_DATA_PATH.exists(), f"Dataset not found: {RAW_DATA_PATH}"
assert 0 < TRAIN_SHARE < 1
assert 0 <= VALIDATION_SHARE < 1
assert TEST_SHARE > 0

# A content-and-policy fingerprint prevents accidental reuse of checkpoints when the
# file at the same path is replaced by a larger dataset or preparation settings change.
dataset_hasher = hashlib.sha256()
with RAW_DATA_PATH.open("rb") as data_handle:
    for chunk in iter(lambda: data_handle.read(1024 * 1024), b""):
        dataset_hasher.update(chunk)
DATASET_SHA256 = dataset_hasher.hexdigest()
policy_payload = {
    "dataset_sha256": DATASET_SHA256,
    "train_share": TRAIN_SHARE,
    "validation_share": VALIDATION_SHARE,
    "min_common_judges": MIN_COMMON_JUDGES,
    "min_margin": MIN_MARGIN,
    "min_confidence": MIN_CONFIDENCE,
    "score_span": SCORE_SPAN,
    "seed": SEED,
}
POLICY_FINGERPRINT = hashlib.sha256(
    json.dumps(policy_payload, sort_keys=True).encode("utf-8")
).hexdigest()[:12]
RUN_ID = f"{RAW_DATA_PATH.stem}-{POLICY_FINGERPRINT}"
RUN_ROOT = Path("outputs") / RUN_ID

print("Dataset:", RAW_DATA_PATH)
print("Dataset size (GB):", round(RAW_DATA_PATH.stat().st_size / 1024**3, 3))
print("Split shares:", {
    "train": TRAIN_SHARE,
    "validation": VALIDATION_SHARE,
    "test": TEST_SHARE,
})
print("Quality thresholds:", {
    "min_common_judges": MIN_COMMON_JUDGES,
    "min_margin": MIN_MARGIN,
    "min_confidence": MIN_CONFIDENCE,
})
print("Dataset SHA-256:", DATASET_SHA256)
print("Run ID:", RUN_ID)


## 5. Build chosen/rejected pairs and assign prompt-level splits

Failed judge blocks are missing evidence, never zero. The split is assigned to each
normalized prompt before its multiple response comparisons are expanded.


In [ ]:
import subprocess

subprocess.run([
    "sdt-build-pairs",
    "--input", str(RAW_DATA_PATH),
    "--output", str(PAIRS_PATH),
    "--report", str(PAIR_REPORT_PATH),
    "--min-common-judges", str(MIN_COMMON_JUDGES),
    "--min-margin", str(MIN_MARGIN),
    "--min-confidence", str(MIN_CONFIDENCE),
    "--score-span", str(SCORE_SPAN),
    "--train-share", str(TRAIN_SHARE),
    "--validation-share", str(VALIDATION_SHARE),
    "--seed", str(SEED),
], check=True)


In [ ]:
import json
from pathlib import Path

pair_report = json.loads(PAIR_REPORT_PATH.read_text())
summary_fields = [
    "records",
    "possible_pairs",
    "retained_pairs",
    "retained_prompts",
    "pairs_by_split",
    "retained_by_split",
    "retained_by_confidence",
    "retained_by_comparison_type",
    "exclusion_reasons",
]
for field in summary_fields:
    print(f"{field}: {pair_report.get(field)}")

retained_by_split = pair_report["retained_by_split"]
for split in ("train", "validation", "test"):
    assert retained_by_split.get(split, 0) > 0, f"No retained {split} pairs"

if not pair_report["methodology"].get("near_duplicate_clustering", False):
    print(
        "\nPRODUCTION NOTE: exact-prompt leakage is controlled, but semantic "
        "near-duplicate clustering is not implemented by pairs.py. Complete that "
        "upstream before treating a large final test split as fully locked."
    )


## 6. Run repository tests

These tests cover failed-judge handling, pair direction, prompt leakage, full-parameter
configuration, Qwen response boundaries, and paired comparison statistics.


In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
    check=True,
)


## 7. Optional: tune hyperparameters using validation only

Leave `RUN_HYPERPARAMETER_TUNING = False` for a single configured run. Set it to `True`
to train the listed candidates, evaluate each candidate and the unchanged baseline on the
same validation pairs, and rank candidates without opening the test split.

For a large dataset, begin with a small, justified grid. A staged pilot may cap training
samples to reduce cost, but the finalists should be confirmed using the complete training
and validation partitions before final testing.


In [ ]:
RUN_HYPERPARAMETER_TUNING = False
FORCE_RETRAIN_TUNING = False

# None uses every retained training pair. A temporary integer cap can be used for
# an engineering pilot, but pilot winners should be confirmed on the full training set.
TUNING_MAX_TRAIN_SAMPLES = None

# Keep most variables fixed so each comparison is interpretable. Edit this list
# deliberately for the full dataset after inspecting update counts and compute cost.
TUNING_CONFIGURATIONS = [
    {
        "name": "lr5e7_ep1_beta010",
        "learning_rate": 5.0e-7,
        "num_train_epochs": 1,
        "warmup_ratio": 0.03,
        "beta": 0.10,
    },
    {
        "name": "lr5e7_ep2_beta010",
        "learning_rate": 5.0e-7,
        "num_train_epochs": 2,
        "warmup_ratio": 0.03,
        "beta": 0.10,
    },
    {
        "name": "lr1e6_ep1_beta010",
        "learning_rate": 1.0e-6,
        "num_train_epochs": 1,
        "warmup_ratio": 0.03,
        "beta": 0.10,
    },
    {
        "name": "lr5e7_ep1_beta005",
        "learning_rate": 5.0e-7,
        "num_train_epochs": 1,
        "warmup_ratio": 0.03,
        "beta": 0.05,
    },
]

# Candidate selection uses unseen validation prompts only. Prompt-macro accuracy
# is primary because several retained pairs can originate from one prompt.
SELECTION_METRIC = "implicit_reward_prompt_macro_accuracy"

print("Hyperparameter tuning enabled:", RUN_HYPERPARAMETER_TUNING)
print("Configurations:")
for configuration in TUNING_CONFIGURATIONS:
    print(" -", configuration)


In [ ]:
import copy
import json
import math
import subprocess
from pathlib import Path

import pandas as pd
import yaml

TUNING_ROOT = RUN_ROOT / "tuning"
TUNING_CONFIG_DIR = Path("configs")
TUNING_ROOT.mkdir(parents=True, exist_ok=True)
TUNING_CONFIG_DIR.mkdir(parents=True, exist_ok=True)

tuning_results = []
SELECTED_CONFIGURATION = None
DPO_MODEL = None
SELECTED_BETA = None

if not RUN_HYPERPARAMETER_TUNING:
    print("Optional validation-only hyperparameter tuning skipped.")
else:
    base_config = yaml.safe_load(BASE_TRAIN_CONFIG.read_text())

    # Score the unchanged baseline once on validation. It is shared by all candidates.
    baseline_validation_summary = TUNING_ROOT / "baseline-validation.json"
    baseline_validation_details = TUNING_ROOT / "baseline-validation-pairs.jsonl"
    subprocess.run([
        "sdt-evaluate-pairs",
        "--pairs", str(PAIRS_PATH),
        "--model", BASELINE_MODEL,
        "--split", "validation",
        "--max-length", str(base_config.get("max_length", 1024)),
        "--output", str(baseline_validation_summary),
        "--details", str(baseline_validation_details),
    ], check=True)

    for specification in TUNING_CONFIGURATIONS:
        name = specification["name"]
        print(f"\n{'=' * 72}\nTUNING CANDIDATE: {name}\n{'=' * 72}")

        candidate_config = copy.deepcopy(base_config)
        candidate_config.update({
            key: value
            for key, value in specification.items()
            if key not in {"name", "warmup_ratio"}
        })
        candidate_config["pairs_file"] = str(PAIRS_PATH.resolve())
        candidate_config["output_dir"] = str(TUNING_ROOT / name / "model")
        candidate_config["max_train_samples"] = TUNING_MAX_TRAIN_SAMPLES
        candidate_config["eval_strategy"] = "epoch"
        candidate_config["save_strategy"] = "epoch"
        candidate_config["seed"] = SEED

        available_train_pairs = int(pair_report["retained_by_split"]["train"])
        used_train_pairs = (
            available_train_pairs
            if TUNING_MAX_TRAIN_SAMPLES is None
            else min(available_train_pairs, int(TUNING_MAX_TRAIN_SAMPLES))
        )
        effective_batch_size = (
            int(candidate_config.get("per_device_train_batch_size", 1))
            * int(candidate_config.get("gradient_accumulation_steps", 8))
        )
        estimated_updates = math.ceil(used_train_pairs / effective_batch_size) * math.ceil(
            float(candidate_config["num_train_epochs"])
        )
        candidate_config["warmup_steps"] = max(
            1,
            round(estimated_updates * float(specification["warmup_ratio"])),
        )

        config_path = TUNING_CONFIG_DIR / f"notebook_tuning_{name}.yaml"
        config_path.write_text(yaml.safe_dump(candidate_config, sort_keys=False))
        model_dir = Path(candidate_config["output_dir"])

        complete_model = (
            (model_dir / "config.json").exists()
            and any(model_dir.glob("*.safetensors"))
        )
        if complete_model and not FORCE_RETRAIN_TUNING:
            resolved_path = model_dir / "resolved_config.json"
            if resolved_path.exists():
                previous = json.loads(resolved_path.read_text())
                for key in ("learning_rate", "num_train_epochs", "warmup_steps", "beta"):
                    assert previous.get(key) == candidate_config.get(key), (
                        f"Existing checkpoint {name} does not match {key}. "
                        "Use a new name or enable FORCE_RETRAIN_TUNING."
                    )
            print("Reusing complete checkpoint:", model_dir)
        else:
            subprocess.run(["sdt-train-dpo", "--config", str(config_path)], check=True)

        candidate_summary = TUNING_ROOT / name / "validation-summary.json"
        candidate_details = TUNING_ROOT / name / "validation-pairs.jsonl"
        comparison_path = TUNING_ROOT / name / "validation-vs-baseline.json"
        relative_details = TUNING_ROOT / name / "validation-relative-pairs.jsonl"
        candidate_summary.parent.mkdir(parents=True, exist_ok=True)

        subprocess.run([
            "sdt-evaluate-pairs",
            "--pairs", str(PAIRS_PATH),
            "--model", str(model_dir),
            "--split", "validation",
            "--max-length", str(candidate_config.get("max_length", 1024)),
            "--output", str(candidate_summary),
            "--details", str(candidate_details),
        ], check=True)
        subprocess.run([
            "sdt-compare-evaluations",
            "--baseline-details", str(baseline_validation_details),
            "--dpo-details", str(candidate_details),
            "--output", str(comparison_path),
            "--details-output", str(relative_details),
            "--beta", str(candidate_config["beta"]),
        ], check=True)

        training_validation = json.loads((model_dir / "validation_metrics.json").read_text())
        comparison = json.loads(comparison_path.read_text())
        primary = comparison["primary_dpo_relative_metrics"]
        secondary = comparison["secondary_absolute_likelihood_metrics"]
        behavior = comparison["checkpoint_behavior_check"]

        tuning_results.append({
            "configuration": name,
            "model_path": str(model_dir),
            "learning_rate": candidate_config["learning_rate"],
            "epochs": candidate_config["num_train_epochs"],
            "warmup_ratio": specification["warmup_ratio"],
            "warmup_steps": candidate_config["warmup_steps"],
            "beta": candidate_config["beta"],
            "max_length": candidate_config.get("max_length", 1024),
            "validation_loss": training_validation.get("validation_loss"),
            "trl_reward_accuracy": training_validation.get("eval_rewards/accuracies"),
            "trl_reward_margin": training_validation.get("eval_rewards/margins"),
            "implicit_reward_accuracy": primary["implicit_reward_accuracy"],
            "implicit_reward_prompt_macro_accuracy": primary[
                "implicit_reward_prompt_macro_accuracy"
            ],
            "mean_implicit_reward_margin": primary["mean_implicit_reward_margin"],
            "absolute_accuracy_delta": secondary["accuracy_delta_dpo_minus_baseline"],
            "absolute_prompt_macro_delta": secondary["prompt_macro_accuracy_delta"],
            "all_validation_pairs_changed": behavior["all_pairs_changed"],
        })

    tuning_table = pd.DataFrame(tuning_results).sort_values(
        by=[SELECTION_METRIC, "mean_implicit_reward_margin", "validation_loss"],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    display(tuning_table)
    tuning_table.to_csv(TUNING_ROOT / "validation-comparison.csv", index=False)

    best = tuning_table.iloc[0].to_dict()
    SELECTED_CONFIGURATION = str(best["configuration"])
    DPO_MODEL = str(best["model_path"])
    SELECTED_BETA = float(best["beta"])
    SELECTED_MAX_LENGTH = int(best["max_length"])
    (TUNING_ROOT / "selected-configuration.json").write_text(
        json.dumps(best, indent=2, default=str) + "\n"
    )

    print("\nSelected using validation only:", SELECTED_CONFIGURATION)
    print("Selected model:", DPO_MODEL)
    print("Selection metric:", SELECTION_METRIC, "=", best[SELECTION_METRIC])
    print(
        "Review the complete table before accepting the automatic ranking. "
        "The locked test split has not been evaluated."
    )


## 8. Train one main configuration or freeze the validation-selected candidate

When optional tuning is disabled, this cell trains the normal full-DPO configuration.
When tuning is enabled, it reuses the candidate selected using validation and does not
train another model.


In [ ]:
import copy
import json
import math
import subprocess
from pathlib import Path

import yaml

if RUN_HYPERPARAMETER_TUNING:
    assert DPO_MODEL is not None and SELECTED_CONFIGURATION is not None
    print("Using validation-selected candidate:", SELECTED_CONFIGURATION)
else:
    main_config = yaml.safe_load(BASE_TRAIN_CONFIG.read_text())
    main_config["pairs_file"] = str(PAIRS_PATH.resolve())
    main_config["output_dir"] = str(RUN_ROOT / "main" / "model")
    main_config["seed"] = SEED
    effective_batch_size = (
        int(main_config.get("per_device_train_batch_size", 1))
        * int(main_config.get("gradient_accumulation_steps", 8))
    )
    estimated_updates = math.ceil(
        int(pair_report["retained_by_split"]["train"]) / effective_batch_size
    ) * math.ceil(float(main_config.get("num_train_epochs", 1)))
    main_config["warmup_steps"] = max(1, round(estimated_updates * 0.03))
    main_config_path = Path("configs/notebook_main.yaml")
    main_config_path.write_text(yaml.safe_dump(main_config, sort_keys=False))

    model_dir = Path(main_config["output_dir"])
    complete_model = (
        (model_dir / "config.json").exists()
        and any(model_dir.glob("*.safetensors"))
    )
    if complete_model:
        print("Reusing complete main checkpoint:", model_dir)
    else:
        subprocess.run(["sdt-train-dpo", "--config", str(main_config_path)], check=True)

    DPO_MODEL = str(model_dir)
    SELECTED_CONFIGURATION = "notebook-main"
    SELECTED_BETA = float(main_config.get("beta", 0.10))
    SELECTED_MAX_LENGTH = int(main_config.get("max_length", 1024))

assert Path(DPO_MODEL, "config.json").exists(), f"Complete model missing: {DPO_MODEL}"
assert any(Path(DPO_MODEL).glob("*.safetensors")), f"Model weights missing: {DPO_MODEL}"
print("Frozen configuration:", SELECTED_CONFIGURATION)
print("Frozen checkpoint:", DPO_MODEL)
print("Frozen beta:", SELECTED_BETA)
print("Frozen max length:", SELECTED_MAX_LENGTH)


## 9. Verify that the frozen checkpoint differs from the baseline

This diagnostic compares a representative model tensor. It confirms that training changed
the checkpoint before any final test interpretation is attempted.


In [ ]:
import subprocess

subprocess.run([
    "sdt-verify-checkpoint",
    "--baseline-model", BASELINE_MODEL,
    "--trained-model", DPO_MODEL,
    "--output", str(RUN_ROOT / "checkpoint-change.json"),
], check=True)


## 10. Explicitly unlock the final test evaluation

Keep this `False` during preparation and hyperparameter tuning. Set it to `True` only after
the configuration, checkpoint, filters, and primary metrics are frozen. Running all cells
with the default value safely skips every test operation.

If test results have already influenced configuration choices, those results are exploratory;
create a new locked test partition for the final full-data experiment.


In [ ]:
RUN_LOCKED_TEST = False

print("Locked test enabled:", RUN_LOCKED_TEST)
if not RUN_LOCKED_TEST:
    print("Test evaluation remains locked. Set RUN_LOCKED_TEST = True only after selection.")


## 11. Evaluate the baseline and frozen DPO checkpoint on the locked test split


In [ ]:
import subprocess
from pathlib import Path

FINAL_TEST_DIR = RUN_ROOT / "final-test"

if not RUN_LOCKED_TEST:
    print("Skipped: locked test is disabled.")
else:
    FINAL_TEST_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "sdt-evaluate-pairs",
        "--pairs", str(PAIRS_PATH),
        "--model", BASELINE_MODEL,
        "--split", "test",
        "--max-length", str(SELECTED_MAX_LENGTH),
        "--output", str(FINAL_TEST_DIR / "baseline-summary.json"),
        "--details", str(FINAL_TEST_DIR / "baseline-pairs.jsonl"),
    ], check=True)
    subprocess.run([
        "sdt-evaluate-pairs",
        "--pairs", str(PAIRS_PATH),
        "--model", DPO_MODEL,
        "--split", "test",
        "--max-length", str(SELECTED_MAX_LENGTH),
        "--output", str(FINAL_TEST_DIR / "dpo-summary.json"),
        "--details", str(FINAL_TEST_DIR / "dpo-pairs.jsonl"),
    ], check=True)


## 12. Verify exact pairing and produce the final comparison

The comparison refuses partial overlap: baseline and DPO detail files must contain exactly
the same pair IDs and token counts.


In [ ]:
import json
import subprocess
from pathlib import Path

if not RUN_LOCKED_TEST:
    print("Skipped: locked test is disabled.")
else:
    def load_details(path):
        return [
            json.loads(line)
            for line in Path(path).read_text().splitlines()
            if line.strip()
        ]

    baseline_rows = load_details(FINAL_TEST_DIR / "baseline-pairs.jsonl")
    dpo_rows = load_details(FINAL_TEST_DIR / "dpo-pairs.jsonl")
    baseline_ids = {str(row["pair_id"]) for row in baseline_rows}
    dpo_ids = {str(row["pair_id"]) for row in dpo_rows}

    print("Baseline evaluated pairs:", len(baseline_ids))
    print("DPO evaluated pairs:", len(dpo_ids))
    print("Shared pair IDs:", len(baseline_ids & dpo_ids))
    assert baseline_ids and baseline_ids == dpo_ids, (
        "Baseline and DPO must contain identical nonempty pair IDs."
    )

    subprocess.run([
        "sdt-compare-evaluations",
        "--baseline-details", str(FINAL_TEST_DIR / "baseline-pairs.jsonl"),
        "--dpo-details", str(FINAL_TEST_DIR / "dpo-pairs.jsonl"),
        "--output", str(FINAL_TEST_DIR / "baseline-vs-dpo.json"),
        "--details-output", str(FINAL_TEST_DIR / "dpo-relative-pairs.jsonl"),
        "--beta", str(SELECTED_BETA),
    ], check=True)


## 13. Display the final locked-test results

DPO-relative implicit reward metrics are primary. Absolute length-normalized preference
accuracy remains a secondary diagnostic. Prompt-level summaries are emphasized because
multiple comparisons from one prompt are correlated.


In [ ]:
import json
import pandas as pd

if not RUN_LOCKED_TEST:
    print("Skipped: locked test is disabled.")
else:
    comparison = json.loads((FINAL_TEST_DIR / "baseline-vs-dpo.json").read_text())
    primary = comparison["primary_dpo_relative_metrics"]
    secondary = comparison["secondary_absolute_likelihood_metrics"]
    behavior = comparison["checkpoint_behavior_check"]

    display(pd.DataFrame([
        {
            "model": "Baseline",
            "absolute_pair_accuracy": secondary["baseline_preference_accuracy"],
            "absolute_prompt_macro_accuracy": secondary["baseline_prompt_macro_accuracy"],
            "mean_absolute_margin": secondary["mean_model_margin_baseline"],
        },
        {
            "model": "Full DPO",
            "absolute_pair_accuracy": secondary["dpo_preference_accuracy"],
            "absolute_prompt_macro_accuracy": secondary["dpo_prompt_macro_accuracy"],
            "mean_absolute_margin": secondary["mean_model_margin_dpo"],
        },
    ]))

    print("\nFROZEN CONFIGURATION:", SELECTED_CONFIGURATION)
    print("\nPRIMARY DPO-RELATIVE RESULTS")
    print(json.dumps(primary, indent=2))
    print("\nSECONDARY ABSOLUTE RESULTS")
    print(json.dumps(secondary, indent=2))
    print("\nCHECKPOINT BEHAVIOR CHECK")
    print(json.dumps(behavior, indent=2))


## 14. Save complete outputs to Google Drive

Colab storage is temporary. Each backup receives a timestamp so previous tuning candidates,
the selected model, validation comparisons, and final reports are preserved.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import shutil
from google.colab import drive

drive.mount("/content/drive")
timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
destination = Path("/content/drive/MyDrive/SDT_full_DPO_results") / timestamp
shutil.copytree("outputs", destination)
print("Saved outputs to:", destination)


## Optional appendix: deliberate tiny-overfit canary

This diagnostic is separate from model selection and never uses the test set. It trains on
eight training pairs and deliberately evaluates those same pairs. It can confirm that the
pipeline is capable of fitting preference examples, but it is not a generalization result.


In [ ]:
RUN_OVERFIT_SANITY = False

if RUN_OVERFIT_SANITY:
    import json
    import subprocess
    from pathlib import Path

    config = Path("configs/sanity_overfit.yaml")
    assert config.exists(), "Missing configs/sanity_overfit.yaml"
    subprocess.run(["sdt-train-dpo", "--config", str(config)], check=True)
    metrics = json.loads(
        Path("outputs/dpo-overfit-sanity/validation_metrics.json").read_text()
    )
    print(json.dumps(metrics, indent=2))
else:
    print("Tiny-overfit canary skipped (normal).")
